# Google Gemma 3 on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Google's Gemma 3 open models, in three sizes that all read text and images. Gemma 4 is
called differently, on another path and with the Responses API, and has its own
notebook, [`01-gemma4-end-to-end`](01-gemma4-end-to-end.ipynb).

| Model | `bedrock-mantle` ID | `bedrock-runtime` ID | Notes |
|---|---|---|---|
| Gemma 3 27B | `google.gemma-3-27b-it` | `google.gemma-3-27b-it` | The default here |
| Gemma 3 12B | `google.gemma-3-12b-it` | `google.gemma-3-12b-it` |  |
| Gemma 3 4B | `google.gemma-3-4b-it` | `google.gemma-3-4b-it` | Smallest and cheapest |

IDs as of September 2026, in `us-east-1`. Which Regions carry each model changes as
models launch, so the setup cell checks your Region.

On `bedrock-mantle` these models serve the OpenAI-compatible **Chat Completions** API at
the bare `/v1` path. On `bedrock-runtime` every size serves **Converse** under the same
ID (section 7).

**Covered:** first call, streaming, multi-turn, structured output, vision, choosing a
size, and Converse on `bedrock-runtime`.

## Setup

Install the pinned requirements first (`pip install -r ../requirements.txt`, Python 3.11
or later). The notebook calls Bedrock with the OpenAI SDK and boto3. From
`_shared/bedrock.py`, this repository's own helper module, it takes an availability
check and a lenient JSON parser, plus the sample images and their known answers for the
vision cells.

Endpoints, authentication and URL paths are explained in
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb).

In [1]:
import json
import sys

import boto3
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

sys.path.insert(0, "../_shared")
from bedrock import endpoints_for, parse_json_lenient

REGION = "us-east-1"
G27B = "google.gemma-3-27b-it"
G12B = "google.gemma-3-12b-it"
G4B = "google.gemma-3-4b-it"

# bedrock-mantle: the OpenAI SDK, with a short-term Bedrock API key minted from your
# AWS credentials. Mint it when you build the client, because the key expires.
client = OpenAI(
    base_url=f"https://bedrock-mantle.{REGION}.api.aws/v1",
    api_key=provide_token(region=REGION),
    timeout=120,
)

# bedrock-runtime: boto3 signs each request with SigV4, so there is no key to mint.
runtime = boto3.client("bedrock-runtime", region_name=REGION)

for model in (G27B, G12B, G4B,):
    served = [name for name, ok in endpoints_for(model, REGION).items() if ok]
    print(f"{model:23} served by: {', '.join(served) or 'neither endpoint in ' + REGION}")

google.gemma-3-27b-it   served by: mantle, runtime
google.gemma-3-12b-it   served by: mantle, runtime
google.gemma-3-4b-it    served by: mantle, runtime


## 1. First call

A single request with a short question. If the model spends the whole `max_tokens`
budget before it answers, `content` is `None` and `finish_reason` is `length`, so the
cell prints both.

In [2]:
response = client.chat.completions.create(
    model=G27B,
    messages=[{"role": "user", "content": "What is the capital of Australia? Answer in one word."}],
    max_tokens=1000,
)
choice = response.choices[0]
print((choice.message.content or "").strip())
print(f"finish_reason={choice.finish_reason}  total_tokens={response.usage.total_tokens}")

Canberra.
finish_reason=stop  total_tokens=26


## 2. Streaming

Set `stream=True` and read `delta.content` from each chunk. Check that a chunk has
choices before indexing it, and read `finish_reason` from the last one: `stop` means the
answer is complete.

In [3]:
stream = client.chat.completions.create(
    model=G27B,
    messages=[{"role": "user", "content": "Count from 1 to 10, separated by commas."}],
    max_tokens=1000,
    stream=True,
)
finish_reason = None
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
    if chunk.choices and chunk.choices[0].finish_reason:
        finish_reason = chunk.choices[0].finish_reason
print(f"\nfinish_reason={finish_reason}")

1, 2, 3, 4, 5, 6,

 7, 8, 9, 10



finish_reason=stop


## 3. Multi-turn

Chat Completions keeps no conversation state on the server, so each request carries the
whole history. Append the assistant's reply before the next user turn.

In [4]:
messages = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "My favourite colour is teal. Reply with one word: noted."},
]
first = client.chat.completions.create(model=G27B, messages=messages, max_tokens=1000)
messages.append({"role": "assistant", "content": first.choices[0].message.content})
messages.append({"role": "user", "content": "What is my favourite colour? One word."})
second = client.chat.completions.create(model=G27B, messages=messages, max_tokens=1000)

print("turn 1:", (first.choices[0].message.content or "").strip())
print("turn 2:", (second.choices[0].message.content or "").strip())

turn 1: Noted.
turn 2: Teal.


## 4. Structured output

`response_format` with a JSON Schema asks the model for output in that shape. Check
`finish_reason` first, because a cut-off answer is cut-off JSON. Parse leniently,
because some models add text after a valid object. A schema controls the shape of the
output and has no say over the values in it, so the cell checks each value against the
source text.

In [5]:
SCHEMA = {
    "type": "object",
    "properties": {
        "invoice_id": {"type": "string"},
        "vendor": {"type": "string"},
        "total": {"type": "number"},
        "currency": {"type": "string"},
    },
    "required": ["invoice_id", "vendor", "total", "currency"],
    "additionalProperties": False,
}
SOURCE = "Invoice: INV-1042\nVendor: Acme Pty Ltd\nAmount due: AUD 1,280.50\nDue date: 30 October 2026"
EXPECTED = {"invoice_id": "INV-1042", "vendor": "Acme Pty Ltd", "total": 1280.5, "currency": "AUD"}

response = client.chat.completions.create(
    model=G27B,
    messages=[{"role": "user", "content": (
        "Extract invoice_id, vendor, total and currency from this invoice as JSON. Write the "
        f"total as a number without a thousands separator, so 2,345.60 becomes 2345.6.\n\n{SOURCE}")}],
    response_format={"type": "json_schema", "json_schema": {"name": "invoice", "strict": True, "schema": SCHEMA}},
    max_tokens=1500,
)
choice = response.choices[0]
if choice.finish_reason != "stop":
    raise RuntimeError(f"incomplete output (finish_reason={choice.finish_reason}); raise max_tokens")

invoice = parse_json_lenient(choice.message.content)
print(json.dumps(invoice, indent=2))
wrong = {k: invoice.get(k) for k, v in EXPECTED.items() if invoice.get(k) != v}
print("values match the source" if not wrong else f"values that do NOT match the source: {wrong}")

{
  "invoice_id": "INV-1042",
  "vendor": "Acme Pty Ltd",
  "total": 1280.5,
  "currency": "AUD"
}
values match the source


## 5. Vision

Send the image as a data URL in an `image_url` content part, next to the text part. The
slide comes from a public AWS talk and ships with this repository, so the answer is
known: its title is `Ingestion from database`, and the cell counts how many of its three
callout lines the model quotes.

In [6]:
from bedrock import SLIDE_CALLOUTS, SLIDE_TITLE, keyword_recall, slide_data_url

response = client.chat.completions.create(
    model=G27B,
    messages=[{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": slide_data_url()}},
        {"type": "text", "text": "Read this slide. Give its title, then quote the three green callout lines."},
    ]}],
    max_tokens=800,
)
text = response.choices[0].message.content or ""
hits, total = keyword_recall(text, SLIDE_CALLOUTS)
print(text.strip()[:500])
print(f"\ntitle found: {SLIDE_TITLE.lower() in text.lower()}   callouts found: {hits}/{total}")

Okay, here's the information from the slide:

**Title:** Ingestion from database

**The three green callout lines are:**

1.  "Pay according to job duration"
2.  "Lower storage cost for rarely accessed data"
3.  "Lower compute cost, up to 90%"

title found: True   callouts found: 3/3


## 6. Choosing a size

The three sizes share one API, so switching is a change of model ID. A common pattern is
to send the request to the smallest model first, check the result, and move up only when
the check fails. The cell reuses the slide from section 5 and stops at the first size
that reads the title and all three callouts.

In [7]:
LADDER = [G4B, G12B, G27B]  # smallest first
question = [{"role": "user", "content": [
    {"type": "image_url", "image_url": {"url": slide_data_url()}},
    {"type": "text", "text": "Read this slide. Give its title, then quote the three green callout lines."},
]}]

for model in LADDER:
    response = client.chat.completions.create(model=model, messages=question, max_tokens=800)
    text = response.choices[0].message.content or ""
    title = SLIDE_TITLE.lower() in text.lower()
    hits, total = keyword_recall(text, SLIDE_CALLOUTS)
    if title and hits == total:
        print(f"{model}: title and {hits}/{total} callouts read; use this size")
        break
    print(f"{model}: title found {title}, callouts {hits}/{total}; trying the next size")

google.gemma-3-4b-it: title and 3/3 callouts read; use this size


## 7. On `bedrock-runtime`: Converse

Every Gemma 3 model is on `bedrock-runtime` under the same ID. Converse has one request
shape for every provider: `content` is a list of blocks, and settings such as the token
limit go in `inferenceConfig`. Read text by block type instead of taking `content[0]`,
so the same code works for models that return other blocks first. The Converse tool loop
is in
[`00-foundations/04`](../00-foundations/04-bedrock-runtime-converse-and-profiles.ipynb).

In [8]:
response = runtime.converse(
    modelId="google.gemma-3-27b-it",
    messages=[{"role": "user", "content": [{"text": "What is the capital of Australia? Answer in one word."}]}],
    inferenceConfig={"maxTokens": 2000},
)
blocks = response["output"]["message"]["content"]
text = "".join(b["text"] for b in blocks if "text" in b)
print("blocks    :", [next(iter(b)) for b in blocks])
print("answer    :", text.strip())
print("stopReason:", response["stopReason"])

blocks    : ['text']
answer    : Canberra.
stopReason: end_turn


## Things that differ

What trips people up on this family, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| Responses API | `POST /v1/responses` returns 400 `The model '…' does not support the '/v1/responses' API` | Use Chat Completions. |
| Path prefix | `/openai/v1/chat/completions` returns 400 ``model `…` isn't supported on this route`` | Use the bare `/v1` path. `/openai/v1` is for the hosted OpenAI GPT, Gemma 4 and Grok models. |
| Conversation state | No `previous_response_id` | Send the full `messages` list on every turn. |
| Empty answer | HTTP 200, `content` is `None`, `finish_reason` is `length` | The model spent `max_tokens` before answering, often on reasoning. Raise the budget and check `finish_reason` before using the text. |
| Structured output values | Valid JSON that holds a wrong value | A schema only controls the shape. Check the values you depend on. |
| Rate limits | 429 under load | Throttling is token-based and there is no requests-per-minute quota. Retry with backoff ([`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)). |
| Metrics | Nothing under `AWS/Bedrock` in CloudWatch | Mantle metrics are in the `AWS/BedrockMantle` namespace. |
| Tool calls | The request with `tools` succeeds, and the model answers in text without calling them (September 2026) | Gemma 3 does not call tools. Use Gemma 4 or another family for tool use. |
| Gemma 4 | Another path (`/openai/v1`), the Responses API, and `bedrock-mantle` only | Call it as [`01-gemma4-end-to-end`](01-gemma4-end-to-end.ipynb) shows. |

## Next

- Gemma 4: [`01-gemma4-end-to-end`](01-gemma4-end-to-end.ipynb)
- Other families on Chat Completions:
  [`04-qwen/01`](../04-qwen/01-qwen3-core-and-tools.ipynb),
  [`07-mistral/01`](../07-mistral/01-mistral-text-and-sizes.ipynb),
  [`10-nvidia-nemotron/01`](../10-nvidia-nemotron/01-nemotron-nano-and-super.ipynb)
- Choosing a model and API across families:
  [`99-cross-cutting/01`](../99-cross-cutting/01-choosing-a-model-and-api.ipynb)
- Quotas, retries and latency:
  [`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)
- Production hardening:
  [`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb)